In [1]:
import bz2
import json

INPUT_FILE = "traceroute-2025-12-10T0000.bz2"
OUTPUT_FILE = "filtered_traceroutes.jsonl"
TARGET_MSM_ID = 88302907


def simplify_traceroute(doc):
    new_doc = {
        "msm_id": doc.get("msm_id"),
        "dst_addr": doc.get("dst_addr"),
        "src_addr": doc.get("src_addr"),
        "prb_addr": doc.get("from"),
        "prb_id": doc.get("prb_id"),
        "result": []
    }

    for hop in doc.get("result", []):
        new_hop = {
            "hop": hop.get("hop"),
            "result": []
        }

        for probe in hop.get("result", []):
            if "from" in probe and "rtt" in probe:
                new_hop["result"].append({
                    "ip_addr": probe["from"],
                    "rtt": probe["rtt"]
                })
            else:
                new_hop["result"].append({"x": "*"})

        new_doc["result"].append(new_hop)

    return new_doc

with bz2.open(INPUT_FILE, "rt", encoding="utf-8") as infile, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:

    for line in infile:
        doc = json.loads(line)

        if doc.get("msm_id") != TARGET_MSM_ID:
            continue
        
        simplified = simplify_traceroute(doc)
        outfile.write(json.dumps(simplified) + "\n")

print("Parser done.")


Parser done.
